In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

def add_noise(image, noise_type='gaussian', param=0.1):
    if len(image.shape) == 3:
        row, col, ch = image.shape
    else:
        row, col = image.shape
        ch = 1

    noisy_img = np.copy(image)

    if noise_type == 'gaussian':
        if ch == 1:
            gauss = np.random.normal(0, param, (row, col))
            noisy_img = image + gauss
        else:
            gauss = np.random.normal(0, param, (row, col, ch))
            noisy_img = image + gauss

    elif noise_type == 'salt_pepper':
        s_vs_p = 0.5
        amount = param

        if ch == 1:
            num_salt = np.ceil(amount * image.size * s_vs_p)
            coords = [np.random.randint(0, i - 1, int(num_salt)) for i in image.shape]
            noisy_img[coords[0], coords[1]] = 1

            num_pepper = np.ceil(amount * image.size * (1. - s_vs_p))
            coords = [np.random.randint(0, i - 1, int(num_pepper)) for i in image.shape]
            noisy_img[coords[0], coords[1]] = 0
        else:
            for i in range(ch):
                num_salt = np.ceil(amount * image[:,:,i].size * s_vs_p)
                coords = [np.random.randint(0, j - 1, int(num_salt)) for j in image[:,:,i].shape]
                noisy_img[coords[0], coords[1], i] = 1

                num_pepper = np.ceil(amount * image[:,:,i].size * (1. - s_vs_p))
                coords = [np.random.randint(0, j - 1, int(num_pepper)) for j in image[:,:,i].shape]
                noisy_img[coords[0], coords[1], i] = 0

    elif noise_type == 'speckle':
        if ch == 1:
            speckle = param * np.random.randn(row, col)
            noisy_img = image + image * speckle
        else:
            speckle = param * np.random.randn(row, col, ch)
            noisy_img = image + image * speckle

    if noisy_img.max() > 1.0:
        noisy_img = np.clip(noisy_img, 0, 255)
    else:
        noisy_img = np.clip(noisy_img, 0, 1.0)

    return noisy_img

def apply_filters(noisy_img):
    kernel_size_small = 3
    kernel_size_large = 5

    mean_filtered_small = cv2.blur(noisy_img, (kernel_size_small, kernel_size_small))
    mean_filtered_large = cv2.blur(noisy_img, (kernel_size_large, kernel_size_large))

    gaussian_filtered_small = cv2.GaussianBlur(noisy_img, (kernel_size_small, kernel_size_small), 0)
    gaussian_filtered_large = cv2.GaussianBlur(noisy_img, (kernel_size_large, kernel_size_large), 0)

    median_filtered_small = cv2.medianBlur(noisy_img.astype(np.uint8), kernel_size_small)
    median_filtered_large = cv2.medianBlur(noisy_img.astype(np.uint8), kernel_size_large)

    blurred = cv2.GaussianBlur(noisy_img, (kernel_size_small, kernel_size_small), 0)

    laplacian = cv2.Laplacian(blurred, cv2.CV_64F)

    laplacian_normalized = np.absolute(laplacian)
    laplacian_normalized = (laplacian_normalized - np.min(laplacian_normalized)) / (np.max(laplacian_normalized) - np.min(laplacian_normalized))

    alpha = 0.5
    laplacian_sharpened = np.clip(blurred + alpha * laplacian, 0, 255).astype(np.uint8)

    return {
        'Mean (3x3)': mean_filtered_small,
        'Mean (5x5)': mean_filtered_large,
        'Gaussian (3x3)': gaussian_filtered_small,
        'Gaussian (5x5)': gaussian_filtered_large,
        'Median (3x3)': median_filtered_small,
        'Median (5x5)': median_filtered_large,
        'Laplacian Edge': laplacian_normalized,
        'Laplacian Sharpened': laplacian_sharpened
    }

def evaluate_filters(original, filtered_images):
    results = {}

    for name, filtered in filtered_images.items():
        if original.dtype != np.uint8:
            original_uint8 = (original * 255).astype(np.uint8) if original.max() <= 1.0 else original.astype(np.uint8)
        else:
            original_uint8 = original

        if filtered.dtype != np.uint8:
            filtered_uint8 = (filtered * 255).astype(np.uint8) if filtered.max() <= 1.0 else filtered.astype(np.uint8)
        else:
            filtered_uint8 = filtered

        if len(original.shape) == 2 or original.shape[2] == 1:
            psnr_value = psnr(original_uint8, filtered_uint8)
            ssim_value = ssim(original_uint8, filtered_uint8)
        else:
            original_gray = cv2.cvtColor(original_uint8, cv2.COLOR_BGR2GRAY)
            filtered_gray = cv2.cvtColor(filtered_uint8, cv2.COLOR_BGR2GRAY)

            psnr_value = psnr(original_uint8, filtered_uint8, data_range=255)
            ssim_value = ssim(original_gray, filtered_gray)

        results[name] = {
            'PSNR': psnr_value,
            'SSIM': ssim_value
        }

    return results

def visualize_results(original, noisy, filtered_images, metrics=None, noise_type=''):
    n_images = len(filtered_images) + 2
    n_cols = 3
    n_rows = (n_images + n_cols - 1) // n_cols

    plt.figure(figsize=(15, 5 * n_rows))

    def prepare_for_display(img):
        if img.dtype != np.uint8:
            img_display = np.clip(img, 0, 255).astype(np.uint8) if img.max() > 1.0 else np.clip(img * 255, 0, 255).astype(np.uint8)
        else:
            img_display = img

        if len(img_display.shape) == 3 and img_display.shape[2] == 3:
            return cv2.cvtColor(img_display, cv2.COLOR_BGR2RGB)
        return img_display

    plt.subplot(n_rows, n_cols, 1)
    original_display = prepare_for_display(original)
    if len(original_display.shape) == 3:
        plt.imshow(original_display)
    else:
        plt.imshow(original_display, cmap='gray')
    plt.title('Original')
    plt.axis('off')

    plt.subplot(n_rows, n_cols, 2)
    noisy_display = prepare_for_display(noisy)
    if len(noisy_display.shape) == 3:
        plt.imshow(noisy_display)
    else:
        plt.imshow(noisy_display, cmap='gray')
    plt.title(f'Noisy ({noise_type})')
    plt.axis('off')

    idx = 3
    for name, filtered in filtered_images.items():
        plt.subplot(n_rows, n_cols, idx)

        filtered_display = prepare_for_display(filtered)
        if len(filtered_display.shape) == 3:
            plt.imshow(filtered_display)
        else:
            plt.imshow(filtered_display, cmap='gray')

        if metrics and name in metrics:
            title = f"{name}\nPSNR: {metrics[name]['PSNR']:.2f} dB\nSSIM: {metrics[name]['SSIM']:.4f}"
        else:
            title = name

        plt.title(title)
        plt.axis('off')
        idx += 1

    plt.tight_layout()
    plt.savefig(f'filtering_results_{noise_type}.png', dpi=300, bbox_inches='tight')
    plt.show()

def plot_comparison_metrics(metrics, noise_types):
    filters = list(metrics[noise_types[0]].keys())
    metric_types = list(metrics[noise_types[0]][filters[0]].keys())

    for metric in metric_types:
        plt.figure(figsize=(12, 8))
        n_filters = len(filters)
        n_noise = len(noise_types)
        width = 0.8 / n_noise

        for i, noise_type in enumerate(noise_types):
            values = [metrics[noise_type][filter_name][metric] for filter_name in filters]

            x = np.arange(n_filters)
            positions = x - 0.4 + (i + 0.5) * width

            plt.bar(positions, values, width=width, label=noise_type)

        plt.xlabel('Filter Type')
        plt.ylabel(metric)
        plt.title(f'Comparison of {metric} across Filters and Noise Types')
        plt.xticks(range(n_filters), filters, rotation=45, ha='right')
        plt.legend()
        plt.grid(True, linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.savefig(f'comparison_{metric}.png', dpi=300, bbox_inches='tight')
        plt.show()

def main():
    image_path = '/content/drive/MyDrive/classroom-behavior/dataset/5k_HRW_yolo_Dataset_jpg/valid/images/0001001.jpg'  # Replace with your image path
    original = cv2.imread(image_path)

    if original is None:
        print(f"Error: Could not read image from {image_path}")
        original = np.ones((300, 300, 3), dtype=np.uint8) * 128
        cv2.rectangle(original, (50, 50), (250, 250), (200, 0, 0), -1)
        cv2.circle(original, (150, 150), 60, (0, 200, 0), -1)
        cv2.line(original, (50, 50), (250, 250), (0, 0, 200), 3)

    gray = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)
    noise_configs = {
        'Gaussian Noise': {'type': 'gaussian', 'param': 25},
        'Salt & Pepper Noise': {'type': 'salt_pepper', 'param': 0.05},
        'Speckle Noise': {'type': 'speckle', 'param': 0.1}
    }

    all_metrics = {}

    for noise_name, config in noise_configs.items():
        print(f"Processing {noise_name}...")

        noisy = add_noise(original, config['type'], config['param'])

        if noisy.dtype != np.uint8:
            noisy = (noisy * 255).astype(np.uint8) if noisy.max() <= 1.0 else noisy.astype(np.uint8)

        filtered_images = apply_filters(noisy)

        metrics = evaluate_filters(original, filtered_images)
        all_metrics[noise_name] = metrics

        visualize_results(original, noisy, filtered_images, metrics, noise_name)

        print(f"Done processing {noise_name}")

        print("\nQuantitative Evaluation:")
        print("-" * 50)
        print(f"{'Filter':<20} | {'PSNR (dB)':<12} | {'SSIM':<12}")
        print("-" * 50)
        for filter_name, metric in metrics.items():
            print(f"{filter_name:<20} | {metric['PSNR']:<12.2f} | {metric['SSIM']:<12.4f}")

    plot_comparison_metrics(all_metrics, list(noise_configs.keys()))

if __name__ == "__main__":
    main()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import open3d as o3d

def load_stereo_images(left_path, right_path):
    img_left = cv2.imread(left_path)
    img_right = cv2.imread(right_path)
    if img_left is None or img_right is None:
        raise ValueError(f"Error loading images from {left_path} and {right_path}")
    if img_left.shape != img_right.shape:
        raise ValueError("Left and right images must have the same dimensions")
    return img_left, img_right

def compute_disparity_block_matching(img_left, img_right, block_size=15, max_disp=64):
    if len(img_left.shape) == 3:
        gray_left = cv2.cvtColor(img_left, cv2.COLOR_BGR2GRAY)
        gray_right = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)
    else:
        gray_left = img_left
        gray_right = img_right
    stereo = cv2.StereoBM_create(numDisparities=max_disp, blockSize=block_size)
    disparity = stereo.compute(gray_left, gray_right)
    disparity_normalized = cv2.normalize(disparity, None, alpha=0, beta=255,
                                        norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    return disparity, disparity_normalized

def compute_disparity_sgbm(img_left, img_right, min_disp=0, max_disp=160, block_size=5):
    max_disp = (max_disp // 16) * 16
    window_size = block_size
    left_matcher = cv2.StereoSGBM_create(
        minDisparity=min_disp,
        numDisparities=max_disp,
        blockSize=block_size,
        P1=8 * 3 * window_size**2,
        P2=32 * 3 * window_size**2,
        disp12MaxDiff=1,
        uniquenessRatio=15,
        speckleWindowSize=100,
        speckleRange=32,
        preFilterCap=63,
        mode=cv2.STEREO_SGBM_MODE_SGBM_3WAY
    )
    disparity = left_matcher.compute(img_left, img_right).astype(np.float32) / 16.0
    disparity_normalized = cv2.normalize(disparity, None, alpha=0, beta=255,
                                        norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    return disparity, disparity_normalized

def reconstruct_3d_point_cloud(disparity, img_left, Q=None, mask_threshold=0):
    h, w = disparity.shape
    if Q is None:
        f = 0.8 * w
        cx, cy = w / 2, h / 2
        baseline = 0.1
        Q = np.array([
            [1, 0, 0, -cx],
            [0, 1, 0, -cy],
            [0, 0, 0, f],
            [0, 0, -1/baseline, 0]
        ])
    mask = disparity > mask_threshold
    points_3d = cv2.reprojectImageTo3D(disparity, Q)
    if len(img_left.shape) == 3:
        colors = cv2.cvtColor(img_left, cv2.COLOR_BGR2RGB)
    else:
        colors = np.stack([img_left, img_left, img_left], axis=2)
    points = points_3d[mask]
    colors = colors[mask]
    return points, colors

def visualize_point_cloud(points, colors):
    pcd = o3d.geometry.PointCloud()
    pcd.points = o3d.utility.Vector3dVector(points)
    pcd.colors = o3d.utility.Vector3dVector(colors / 255.0)
    pcd, _ = pcd.remove_statistical_outlier(nb_neighbors=20, std_ratio=2.0)
    o3d.io.write_point_cloud("point_cloud.ply", pcd)
    o3d.visualization.draw_geometries([pcd])
    return pcd

def visualize_point_cloud_matplotlib(points, colors, step=100):
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')
    points_downsampled = points[::step]
    colors_downsampled = colors[::step]
    colors_normalized = colors_downsampled / 255.0
    ax.scatter(
        points_downsampled[:, 0],
        points_downsampled[:, 1],
        points_downsampled[:, 2],
        c=colors_normalized,
        s=1
    )
    ax.set_xlabel('X')
    ax.set_ylabel('Y')
    ax.set_zlabel('Z')
    ax.view_init(elev=-70, azim=-90)
    plt.savefig('point_cloud_matplotlib.png', dpi=300, bbox_inches='tight')
    plt.show()

def find_keypoints_and_matches(img_left, img_right):
    if len(img_left.shape) == 3:
        gray_left = cv2.cvtColor(img_left, cv2.COLOR_BGR2GRAY)
        gray_right = cv2.cvtColor(img_right, cv2.COLOR_BGR2GRAY)
    else:
        gray_left = img_left
        gray_right = img_right
    sift = cv2.SIFT_create()
    kp_left, des_left = sift.detectAndCompute(gray_left, None)
    kp_right, des_right = sift.detectAndCompute(gray_right, None)
    FLANN_INDEX_KDTREE = 1
    index_params = dict(algorithm=FLANN_INDEX_KDTREE, trees=5)
    search_params = dict(checks=50)
    flann = cv2.FlannBasedMatcher(index_params, search_params)
    matches = flann.knnMatch(des_left, des_right, k=2)
    good_matches = []
    for m, n in matches:
        if m.distance < 0.7 * n.distance:
            good_matches.append(m)
    return kp_left, kp_right, good_matches

def estimate_fundamental_matrix(kp_left, kp_right, matches):
    if len(matches) < 8:
        print("Not enough matches to compute fundamental matrix (need at least 8)")
        F = np.eye(3)
        return F, np.array([]), np.array([])
    pts_left = np.float32([kp_left[m.queryIdx].pt for m in matches])
    pts_right = np.float32([kp_right[m.trainIdx].pt for m in matches])
    try:
        F, mask = cv2.findFundamentalMat(pts_left, pts_right, cv2.FM_RANSAC,
                                      ransacReprojThreshold=3.0, confidence=0.99)
        if F is None or mask is None:
            print("Could not compute a valid fundamental matrix")
            F = np.eye(3)
            return F, np.array([]), np.array([])
        inlier_mask = mask.ravel() == 1
        pts_left_inliers = pts_left[inlier_mask]
        pts_right_inliers = pts_right[inlier_mask]
        if len(pts_left_inliers) < 8:
            print(f"Too few inliers ({len(pts_left_inliers)}) to compute a reliable fundamental matrix")
    except Exception as e:
        print(f"Error computing fundamental matrix: {e}")
        F = np.eye(3)
        return F, np.array([]), np.array([])
    return F, pts_left_inliers, pts_right_inliers

def draw_epipolar_lines(img_left, img_right, pts_left, pts_right, F, num_lines=10):
    h, w = img_left.shape[:2]
    img_left_lines = img_left.copy()
    img_right_lines = img_right.copy()
    if len(pts_left) == 0 or len(pts_right) == 0:
        print("No inliers to draw epipolar lines. Showing original images instead.")
        text = "No valid inliers found for epipolar lines"
        font = cv2.FONT_HERSHEY_SIMPLEX
        textsize = cv2.getTextSize(text, font, 1, 2)[0]
        textX = (img_left.shape[1] - textsize[0]) // 2
        textY = (img_left.shape[0] + textsize[1]) // 2
        cv2.putText(img_left_lines, text, (textX, textY), font, 1, (0, 0, 255), 2)
        cv2.putText(img_right_lines, text, (textX, textY), font, 1, (0, 0, 255), 2)
        img_epipolar = np.hstack((img_left_lines, img_right_lines))
        cv2.imwrite('epipolar_lines.png', img_epipolar)
        plt.figure(figsize=(15, 7))
        plt.imshow(cv2.cvtColor(img_epipolar, cv2.COLOR_BGR2RGB))
        plt.title('Original Images (No Epipolar Lines)')
        plt.axis('off')
        plt.tight_layout()
        plt.savefig('epipolar_lines_plt.png', dpi=300, bbox_inches='tight')
        plt.show()
        return img_epipolar
    if len(pts_left) > num_lines:
        indices = np.linspace(0, len(pts_left) - 1, num_lines).astype(int)
        pts_left_subset = pts_left[indices]
        pts_right_subset = pts_right[indices]
    else:
        pts_left_subset = pts_left
        pts_right_subset = pts_right
    for pt in pts_left_subset:
        cv2.circle(img_left_lines, (int(pt[0]), int(pt[1])), 5, (0, 0, 255), -1)
    for pt in pts_right_subset:
        cv2.circle(img_right_lines, (int(pt[0]), int(pt[1])), 5, (0, 0, 255), -1)
    try:
        lines_right = cv2.computeCorrespondEpilines(pts_left_subset.reshape(-1, 1, 2), 1, F)
        lines_right = lines_right.reshape(-1, 3)
        lines_left = cv2.computeCorrespondEpilines(pts_right_subset.reshape(-1, 1, 2), 2, F)
        lines_left = lines_left.reshape(-1, 3)
        for i, (pt_left, pt_right) in enumerate(zip(pts_left_subset, pts_right_subset)):
            try:
                color = np.random.randint(0, 255, 3).tolist()
                right_line = lines_right[i]
                if abs(right_line[1]) > 1e-5:
                    x0, y0 = 0, int(-right_line[2] / right_line[1])
                    x1, y1 = w, int(-(right_line[2] + right_line[0] * w) / right_line[1])
                    if 0 <= y0 <= h and 0 <= y1 <= h:
                        img_right_lines = cv2.line(img_right_lines, (x0, y0), (x1, y1), color, 1)
                left_line = lines_left[i]
                if abs(left_line[1]) > 1e-5:
                    x0, y0 = 0, int(-left_line[2] / left_line[1])
                    x1, y1 = w, int(-(left_line[2] + left_line[0] * w) / left_line[1])
                    if 0 <= y0 <= h and 0 <= y1 <= h:
                        img_left_lines = cv2.line(img_left_lines, (x0, y0), (x1, y1), color, 1)
            except Exception as e:
                print(f"Error drawing epipolar line {i}: {e}")
                continue
    except Exception as e:
        print(f"Error computing epipolar lines: {e}")
    img_epipolar = np.hstack((img_left_lines, img_right_lines))
    cv2.imwrite('epipolar_lines.png', img_epipolar)
    plt.figure(figsize=(15, 7))
    plt.imshow(cv2.cvtColor(img_epipolar, cv2.COLOR_BGR2RGB))
    plt.title('Epipolar Lines')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('epipolar_lines_plt.png', dpi=300, bbox_inches='tight')
    plt.show()
    return img_epipolar

def visualize_matches(img_left, img_right, kp_left, kp_right, matches, mask=None):
    img_matches = cv2.drawMatches(img_left, kp_left, img_right, kp_right, matches, None,
                                 matchColor=(0, 255, 0), singlePointColor=(255, 0, 0),
                                 matchesMask=mask, flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    cv2.imwrite('feature_matches.png', img_matches)
    plt.figure(figsize=(15, 7))
    plt.imshow(cv2.cvtColor(img_matches, cv2.COLOR_BGR2RGB))
    plt.title('Feature Matches')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('feature_matches_plt.png', dpi=300, bbox_inches='tight')
    plt.show()
    return img_matches

def rectify_stereo_images(img_left, img_right, F):
    h, w = img_left.shape[:2]
    _, H1, H2 = cv2.stereoRectifyUncalibrated(
        np.float32(np.column_stack(np.where(np.ones((h, w)) > 0)[::-1])),
        np.float32(np.column_stack(np.where(np.ones((h, w)) > 0)[::-1])),
        F, (w, h)
    )
    img_left_rectified = cv2.warpPerspective(img_left, H1, (w, h))
    img_right_rectified = cv2.warpPerspective(img_right, H2, (w, h))
    img_rectified = np.hstack((img_left_rectified, img_right_rectified))
    cv2.imwrite('rectified_stereo.png', img_rectified)
    plt.figure(figsize=(15, 7))
    plt.imshow(cv2.cvtColor(img_rectified, cv2.COLOR_BGR2RGB))
    plt.title('Rectified Stereo Images')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('rectified_stereo_plt.png', dpi=300, bbox_inches='tight')
    plt.show()
    return img_left_rectified, img_right_rectified

def evaluate_disparity_map(disparity, ground_truth=None):
    if ground_truth is None:
        print("No ground truth disparity map provided for evaluation.")
        return None
    if ground_truth.max() > 255:
        ground_truth_norm = cv2.normalize(ground_truth, None, alpha=0, beta=255,
                                       norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    else:
        ground_truth_norm = ground_truth
    if disparity.shape != ground_truth_norm.shape:
        print(f"Error: Disparity ({disparity.shape}) and ground truth ({ground_truth_norm.shape}) shapes don't match.")
        return None
    mae = np.mean(np.abs(disparity - ground_truth_norm))
    rmse = np.sqrt(np.mean((disparity - ground_truth_norm) ** 2))
    threshold = 5
    bpr = np.sum(np.abs(disparity - ground_truth_norm) > threshold) / disparity.size
    results = {
        'MAE': mae,
        'RMSE': rmse,
        'BPR': bpr
    }
    print("Disparity Map Evaluation Metrics:")
    print(f"Mean Absolute Error (MAE): {mae:.2f}")
    print(f"Root Mean Square Error (RMSE): {rmse:.2f}")
    print(f"Bad Pixel Ratio (BPR) at threshold {threshold}: {bpr:.4f}")
    return results

def compare_disparity_methods(img_left, img_right):
    disparity_bm, disparity_bm_norm = compute_disparity_block_matching(
        img_left, img_right, block_size=15, max_disp=64
    )
    disparity_sgbm, disparity_sgbm_norm = compute_disparity_sgbm(
        img_left, img_right, min_disp=0, max_disp=160, block_size=5
    )
    plt.figure(figsize=(15, 7))
    plt.subplot(1, 2, 1)
    plt.imshow(disparity_bm_norm, cmap='plasma')
    plt.title('Block Matching Disparity')
    plt.axis('off')
    plt.subplot(1, 2, 2)
    plt.imshow(disparity_sgbm_norm, cmap='plasma')
    plt.title('SGBM Disparity')
    plt.axis('off')
    plt.tight_layout()
    plt.savefig('disparity_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    return (disparity_bm, disparity_bm_norm), (disparity_sgbm, disparity_sgbm_norm)

def create_synthetic_stereo_pair(width=600, height=400):
    gradient = np.linspace(150, 220, width).astype(np.uint8)
    img_left = np.zeros((height, width, 3), dtype=np.uint8)
    img_right = np.zeros((height, width, 3), dtype=np.uint8)
    for i in range(height):
        img_left[i, :, :] = gradient.reshape(1, -1, 1)
        img_right[i, :, :] = gradient.reshape(1, -1, 1)
    grid_step = 50
    grid_color = (130, 130, 130)
    grid_thickness = 1
    for x in range(0, width, grid_step):
        cv2.line(img_left, (x, 0), (x, height), grid_color, grid_thickness)
        cv2.line(img_right, (x, 0), (x, height), grid_color, grid_thickness)
    for y in range(0, height, grid_step):
        cv2.line(img_left, (0, y), (width, y), grid_color, grid_thickness)
        cv2.line(img_right, (0, y), (width, y), grid_color, grid_thickness)
    np.random.seed(42)
    shapes = [
        {'type': 'rectangle', 'center': (150, 150), 'size': (80, 80), 'color': (255, 0, 0), 'disparity': 25},
        {'type': 'circle', 'center': (400, 200), 'radius': 60, 'color': (0, 0, 255), 'disparity': 15},
        {'type': 'triangle', 'vertices': np.array([[300, 100], [350, 200], [250, 200]]), 'color': (0, 255, 0), 'disparity': 35},
        {'type': 'rectangle', 'center': (450, 300), 'size': (60, 60), 'color': (255, 255, 0), 'disparity': 20},
        {'type': 'circle', 'center': (200, 300), 'radius': 40, 'color': (255, 0, 255), 'disparity': 10}
    ]
    for shape in shapes:
        if shape['type'] == 'rectangle':
            x, y = shape['center']
            w, h = shape['size']
            disp = shape['disparity']
            pt1_left = (x - w//2, y - h//2)
            pt2_left = (x + w//2, y + h//2)
            cv2.rectangle(img_left, pt1_left, pt2_left, shape['color'], -1)
            pt1_right = (x - w//2 - disp, y - h//2)
            pt2_right = (x + w//2 - disp, y + h//2)
            cv2.rectangle(img_right, pt1_right, pt2_right, shape['color'], -1)
        elif shape['type'] == 'circle':
            x, y = shape['center']
            r = shape['radius']
            disp = shape['disparity']
            cv2.circle(img_left, (x, y), r, shape['color'], -1)
            cv2.circle(img_right, (x - disp, y), r, shape['color'], -1)
        elif shape['type'] == 'triangle':
            vertices = shape['vertices']
            disp = shape['disparity']
            cv2.fillPoly(img_left, [vertices], shape['color'])
            vertices_right = vertices.copy()
            vertices_right[:, 0] -= disp
            cv2.fillPoly(img_right, [vertices_right], shape['color'])
    for _ in range(30):
        x = np.random.randint(50, width-50)
        y = np.random.randint(50, height-50)
        r = np.random.randint(3, 8)
        color = tuple(np.random.randint(0, 255, 3).tolist())
        disp = np.random.randint(5, 30)
        cv2.circle(img_left, (x, y), r, color, -1)
        cv2.circle(img_right, (x - disp, y), r, color, -1)
    noise_left = np.random.randint(0, 15, (height, width, 3), dtype=np.int16)
    noise_right = np.random.randint(0, 15, (height, width, 3), dtype=np.int16)
    img_left = np.clip(img_left.astype(np.int16) + noise_left, 0, 255).astype(np.uint8)
    img_right = np.clip(img_right.astype(np.int16) + noise_right, 0, 255).astype(np.uint8)
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(img_left, 'Left Image', (20, 30), font, 1, (0, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(img_right, 'Right Image', (20, 30), font, 1, (0, 0, 0), 2, cv2.LINE_AA)
    cv2.imwrite('synthetic_left.png', img_left)
    cv2.imwrite('synthetic_right.png', img_right)
    return img_left, img_right

def main():
    try_filenames = [
        ('left.jpg', 'right.jpg'),
        ('im0.png', 'im1.png'),
        ('left.png', 'right.png'),
        ('stereo_left.jpg', 'stereo_right.jpg')
    ]
    img_left = None
    img_right = None
    for left_name, right_name in try_filenames:
        try:
            img_left, img_right = load_stereo_images(left_name, right_name)
            print(f"Loaded images from '{left_name}' and '{right_name}'")
            break
        except Exception:
            continue
    if img_left is None or img_right is None:
        print("No stereo images found. Creating synthetic stereo pair...")
        img_left, img_right = create_synthetic_stereo_pair(600, 400)
    print("Finding keypoints and matches...")
    kp_left, kp_right, matches = find_keypoints_and_matches(img_left, img_right)
    print(f"Found {len(kp_left)} keypoints in left image, {len(kp_right)} in right image")
    print(f"Found {len(matches)} good matches")
    visualize_matches(img_left, img_right, kp_left, kp_right, matches[:100])
    print("Estimating fundamental matrix...")
    F, pts_left_inliers, pts_right_inliers = estimate_fundamental_matrix(kp_left, kp_right, matches)
    print(f"Fundamental matrix:\n{F}")
    print(f"Found {len(pts_left_inliers)} inlier matches")
    print("Drawing epipolar lines...")
    draw_epipolar_lines(img_left, img_right, pts_left_inliers, pts_right_inliers, F, num_lines=15)
    print("Computing disparity maps...")
    (disparity_bm, disparity_bm_norm), (disparity_sgbm, disparity_sgbm_norm) = compare_disparity_methods(img_left, img_right)
    print("Reconstructing 3D point cloud...")
    points, colors = reconstruct_3d_point_cloud(disparity_sgbm, img_left, mask_threshold=5)
    try:
        print("Visualizing point cloud with Open3D...")
        visualize_point_cloud(points, colors)
    except Exception as e:
        print(f"Error using Open3D: {e}")
        print("Falling back to Matplotlib for point cloud visualization...")
        visualize_point_cloud_matplotlib(points, colors)
    print("Rectifying stereo images...")
    img_left_rectified, img_right_rectified = rectify_stereo_images(img_left, img_right, F)
    print("Computing disparity maps on rectified images...")
    compare_disparity_methods(img_left_rectified, img_right_rectified)
    print("Stereo reconstruction pipeline completed!")

if __name__ == "__main__":
    main()

Output hidden; open in https://colab.research.google.com to view.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import os
import glob

def load_images(folder_path):
    for ext in ["jpg", "png", "jpeg", "JPG", "PNG"]:
        image_paths = sorted(glob.glob(os.path.join(folder_path, f"*.{ext}")))
        if image_paths:
            break
    if not image_paths:
        print(f"No images found in {folder_path}")
        return []
    images = []
    for path in image_paths:
        img = cv2.imread(path)
        if img is not None:
            images.append(img)
            print(f"Loaded image: {path}, shape: {img.shape}")
    return images

def create_test_images():
    base = np.zeros((400, 1200, 3), dtype=np.uint8)
    for i in range(1200):
        value = int(180 + 75 * np.sin(i * np.pi / 600))
        base[:, i] = [value, value, value]
    cv2.rectangle(base, (500, 150), (700, 250), (0, 0, 255), -1)
    cv2.circle(base, (300, 200), 70, (255, 0, 0), -1)
    cv2.circle(base, (900, 200), 70, (0, 255, 0), -1)
    font = cv2.FONT_HERSHEY_SIMPLEX
    cv2.putText(base, "PANORAMA", (550, 300), font, 1, (255, 255, 255), 2)
    cv2.putText(base, "LEFT", (250, 300), font, 1, (255, 255, 255), 2)
    cv2.putText(base, "RIGHT", (850, 300), font, 1, (255, 255, 255), 2)
    images = []
    width = 400
    for i in range(4):
        start_x = i * 200
        img = base[:, start_x:start_x+width].copy()
        cv2.putText(img, f"Image {i+1}", (10, 30), font, 1, (255, 255, 255), 2)
        images.append(img)
        cv2.imwrite(f"test_image_{i+1}.jpg", img)
    cv2.imwrite("full_panorama.jpg", base)
    return images

def detect_and_match_features(img1, img2, method="sift"):
    gray1 = cv2.cvtColor(img1, cv2.COLOR_BGR2GRAY)
    gray2 = cv2.cvtColor(img2, cv2.COLOR_BGR2GRAY)
    if method.lower() == "sift":
        detector = cv2.SIFT_create()
    elif method.lower() == "orb":
        detector = cv2.ORB_create(nfeatures=2000)
    else:
        print(f"Unknown method: {method}. Using SIFT.")
        detector = cv2.SIFT_create()
    kp1, des1 = detector.detectAndCompute(gray1, None)
    kp2, des2 = detector.detectAndCompute(gray2, None)
    print(f"Detected {len(kp1)} keypoints in image 1 and {len(kp2)} in image 2")
    if method.lower() == "orb":
        matcher = cv2.BFMatcher(cv2.NORM_HAMMING, crossCheck=False)
    else:
        matcher = cv2.BFMatcher()
    raw_matches = matcher.knnMatch(des1, des2, k=2)
    good_matches = []
    for m, n in raw_matches:
        if m.distance < 0.75 * n.distance:
            good_matches.append(m)
    print(f"Found {len(good_matches)} good matches out of {len(raw_matches)} total matches")
    return kp1, kp2, good_matches

def find_homography(kp1, kp2, matches):
    if len(matches) < 4:
        print(f"Not enough matches to estimate homography: {len(matches)} < 4")
        return None, []
    src_pts = np.float32([kp1[m.queryIdx].pt for m in matches]).reshape(-1, 1, 2)
    dst_pts = np.float32([kp2[m.trainIdx].pt for m in matches]).reshape(-1, 1, 2)
    H, mask = cv2.findHomography(src_pts, dst_pts, cv2.RANSAC, 4.0)
    if mask is None:
        print("No valid homography found")
        return None, []
    matches_mask = mask.ravel().tolist()
    inliers = [matches[i] for i, mask_bit in enumerate(matches_mask) if mask_bit]
    print(f"Found {sum(matches_mask)} inliers out of {len(matches)} matches")
    return H, inliers

def visualize_matches(img1, kp1, img2, kp2, matches, title="Matches"):
    if len(matches) > 100:
        matches = matches[:100]
    match_img = cv2.drawMatches(img1, kp1, img2, kp2, matches, None,
                              matchColor=(0, 255, 0), singlePointColor=(255, 0, 0),
                              flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS)
    plt.figure(figsize=(12, 6))
    plt.imshow(cv2.cvtColor(match_img, cv2.COLOR_BGR2RGB))
    plt.title(title)
    plt.tight_layout()
    plt.savefig(f"{title.replace(' ', '_').lower()}.jpg", dpi=300)
    plt.show()
    return match_img

def stitch_images(images, method="sift"):
    if len(images) < 2:
        print("Need at least 2 images to stitch")
        return None
    result = images[0]
    for i in range(1, len(images)):
        print(f"\nStitching image {i+1}/{len(images)}")
        img = images[i]
        kp1, kp2, matches = detect_and_match_features(result, img, method)
        visualize_matches(result, kp1, img, kp2, matches, f"Matches {i}")
        H, inliers = find_homography(kp1, kp2, matches)
        if H is None:
            print(f"Failed to find homography between result and image {i+1}")
            continue
        h1, w1 = result.shape[:2]
        h2, w2 = img.shape[:2]
        pts = np.float32([[0, 0], [0, h1], [w1, h1], [w1, 0]]).reshape(-1, 1, 2)
        dst = cv2.perspectiveTransform(pts, H)
        dst_corners = np.concatenate((dst, np.float32([[0, 0], [0, h2], [w2, h2], [w2, 0]]).reshape(-1, 1, 2)))
        [xmin, ymin] = np.int32(dst_corners.min(axis=0).ravel() - 0.5)
        [xmax, ymax] = np.int32(dst_corners.max(axis=0).ravel() + 0.5)
        t = [-xmin, -ymin]
        Ht = np.array([[1, 0, t[0]], [0, 1, t[1]], [0, 0, 1]])
        result_warped = cv2.warpPerspective(result, Ht.dot(H), (xmax-xmin, ymax-ymin))
        img_offset = np.zeros_like(result_warped)
        img_offset[t[1]:h2+t[1], t[0]:w2+t[0]] = img
        mask = np.zeros_like(result_warped)
        mask[t[1]:h2+t[1], t[0]:w2+t[0]] = 255
        result = np.where(mask == 0, result_warped, img_offset)
        cv2.imwrite(f"panorama_step_{i}.jpg", result)
    return result

def compare_methods(images):
    print("\n===== Stitching with SIFT =====")
    panorama_sift = stitch_images(images, "sift")
    if panorama_sift is not None:
        cv2.imwrite("panorama_sift.jpg", panorama_sift)
        plt.figure(figsize=(12, 6))
        plt.imshow(cv2.cvtColor(panorama_sift, cv2.COLOR_BGR2RGB))
        plt.title("Panorama with SIFT")
        plt.savefig("panorama_sift_plt.jpg", dpi=300)
        plt.show()
    print("\n===== Stitching with ORB =====")
    panorama_orb = stitch_images(images, "orb")
    if panorama_orb is not None:
        cv2.imwrite("panorama_orb.jpg", panorama_orb)
        plt.figure(figsize=(12, 6))
        plt.imshow(cv2.cvtColor(panorama_orb, cv2.COLOR_BGR2RGB))
        plt.title("Panorama with ORB")
        plt.savefig("panorama_orb_plt.jpg", dpi=300)
        plt.show()
    if panorama_sift is not None and panorama_orb is not None:
        plt.figure(figsize=(15, 10))
        plt.subplot(2, 1, 1)
        plt.imshow(cv2.cvtColor(panorama_sift, cv2.COLOR_BGR2RGB))
        plt.title("Panorama with SIFT")
        plt.axis('off')
        plt.subplot(2, 1, 2)
        plt.imshow(cv2.cvtColor(panorama_orb, cv2.COLOR_BGR2RGB))
        plt.title("Panorama with ORB")
        plt.axis('off')
        plt.tight_layout()
        plt.savefig("panorama_comparison.jpg", dpi=300)
        plt.show()

def main():
    images = []
    folders = ["stit", "images", "."]
    for folder in folders:
        if os.path.exists(folder):
            images = load_images(folder)
            if len(images) >= 2:
                print(f"Found {len(images)} images in folder '{folder}'")
                break
    if len(images) < 2:
        print("No suitable images found. Creating test images...")
        images = create_test_images()
    compare_methods(images)
    print("\nImage stitching pipeline completed!")

if __name__ == "__main__":
    main()

Output hidden; open in https://colab.research.google.com to view.